# 16. Four-cell example inspection

Dr Hunter, 22 October 2026:

> Compare some of the ones that it's getting right and ones that it's getting wrong. Is there
> anything that you can see why it's getting it wrong? ... some that it got right, some that
> got wrong, not-hate, and some that got right, some that got wrong on hate, and see if
> there's anything distinct, particularly for the hate ones.

He made the same point with an analogy about insect identification: when a classifier calls
a hoverfly a wasp, the useful question is whether the hoverfly actually does look like a
wasp. The equivalent here is whether the missed hate posts actually read like not-hate posts.

The earlier error analysis only looked at **one** of the four cells, the missed hate posts.
This notebook samples all four, so the comparison he asked for is possible:

|  | Predicted not-hate | Predicted hate |
|---|---|---|
| **Actually not-hate** | correct, cell TN | false alarm, cell FP |
| **Actually hate** | **missed, cell FN** | caught, cell TP |

It also samples the cross-dataset test set, because he asked for the same exercise on a
second corpus.

**No GPU, no training.** It reads the saved per-example probabilities in
`writing/probabilities/` and writes a spreadsheet to read by hand. Runs in seconds.

Sampling is stratified by model confidence, so each cell contains confident, middling and
borderline cases rather than only the extremes.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path(os.environ.get('HINGLISH_HATE_ROOT', r'H:\Data\msc-hinglish-hate'))
if not (ROOT / 'data').exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'notebooks'))

W    = ROOT / 'writing'
PROB = W / 'probabilities'
SEED = 42
PER_CELL = 15          # how many examples per cell; 15 x 4 cells = 60 posts per model

MODELS = [('tfidf_logreg',            'TF-IDF + LogReg'),
          ('xlm_roberta_base',        'XLM-RoBERTa'),
          ('google_muril_base_cased', 'MuRIL'),
          ('ai4bharat_indic_bert',    'IndicBERT v1')]

CELLS = {
    (1, 1): 'TP caught hate',
    (1, 0): 'FN missed hate',
    (0, 0): 'TN correct not-hate',
    (0, 1): 'FP false alarm',
}
print('root:', ROOT, '| probabilities found:', len(list(PROB.glob('*.csv'))))

## Lexicon hits, so the reading is not purely impressionistic

Each sampled post is annotated with the number of lexicon terms it contains, using the same
tokeniser the feature pipeline uses. That makes one hypothesis checkable while reading:
whether the missed posts are simply the ones with no slur in them.

In [ ]:
from hinglish_hate.features import TOKEN, load_lexicon
from hinglish_hate.preprocessing import clean_text

TERMS = set(load_lexicon(ROOT / 'data' / 'hinglish_slur_lexicon.csv'))
print(f'{len(TERMS)} lexicon surface forms loaded')

def lex_hits(text):
    toks = TOKEN.findall(clean_text(str(text)))
    return sum(1 for t in toks if t in TERMS)

def n_words(text):
    return len(TOKEN.findall(clean_text(str(text))))

In [ ]:
def sample_cells(tag, name, split, per_cell=PER_CELL, seed=SEED):
    """Stratified sample from each of the four confusion-matrix cells."""
    path = PROB / f'{tag}__{split}.csv'
    if not path.exists():
        print(f'  missing: {path.name}')
        return pd.DataFrame()

    d = pd.read_csv(path)
    d['pred']  = (d['p_hate'] >= 0.5).astype(int)
    d['D']     = (d[['p_not', 'p_hate']].max(axis=1) - d[['p_not', 'p_hate']].min(axis=1)).round(3)
    d['cell']  = [CELLS[(a, p)] for a, p in zip(d['label'], d['pred'])]
    d['model'] = name
    d['split'] = split

    out = []
    for cell, grp in d.groupby('cell'):
        if grp.empty:
            continue
        # stratify by confidence: borderline, middling and confident, sampled evenly
        g = grp.sort_values('D').reset_index(drop=True)
        g['band'] = pd.qcut(g['D'].rank(method='first'), q=min(3, len(g)),
                            labels=False, duplicates='drop')
        take = max(1, per_cell // 3)
        picked = (g.groupby('band', group_keys=False)
                   .apply(lambda t: t.sample(min(take, len(t)), random_state=seed)))
        out.append(picked.head(per_cell))

    s = pd.concat(out, ignore_index=True) if out else pd.DataFrame()
    if s.empty:
        return s
    s['lex_hits'] = s['text'].map(lex_hits)
    s['n_words']  = s['text'].map(n_words)
    s['your_verdict'] = ''      # fill in by hand: hate / not hate / unsure
    s['your_note']    = ''      # fill in by hand: why
    cols = ['model', 'split', 'cell', 'label', 'pred', 'p_hate', 'D',
            'lex_hits', 'n_words', 'your_verdict', 'your_note', 'text']
    return s[cols]


frames = []
for split in ['within', 'cross']:
    for tag, name in MODELS:
        f = sample_cells(tag, name, split)
        if not f.empty:
            frames.append(f)
            print(f'{name:18s} {split:7s} -> {len(f)} posts')

samples = pd.concat(frames, ignore_index=True)
print(f'\ntotal sampled: {len(samples)} rows')

## The aggregate check, before reading a single post

If the missed hate posts really do look like not-hate posts, that should show up in the
numbers as well as on the page. Three quantities worth comparing across the four cells:

- **lexicon hits** — do the missed posts simply contain no slur?
- **length** — are they too short to classify?
- **model confidence** — is the model unsure about them, or confidently wrong?

This is the quantitative half of Hunter's question. The reading is the other half.

In [ ]:
agg = (samples[samples.split == 'within']
       .groupby(['model', 'cell'])
       .agg(n=('text', 'size'),
            mean_lex_hits=('lex_hits', 'mean'),
            pct_zero_lex=('lex_hits', lambda s: 100 * (s == 0).mean()),
            mean_words=('n_words', 'mean'),
            mean_D=('D', 'mean'))
       .round(2).reset_index())
display(agg)

print('\nThe comparison that answers his question, within-dataset:')
for m in agg['model'].unique():
    a = agg[agg.model == m].set_index('cell')
    if 'FN missed hate' in a.index and 'TP caught hate' in a.index:
        fn, tp = a.loc['FN missed hate'], a.loc['TP caught hate']
        print(f'  {m:18s} missed {fn.mean_lex_hits:.2f} lexicon hits / {fn.pct_zero_lex:.0f}% with none'
              f'   vs caught {tp.mean_lex_hits:.2f} / {tp.pct_zero_lex:.0f}%')

In [ ]:
# full-population version of the same comparison, not just the sample
rows = []
for split in ['within', 'cross']:
    for tag, name in MODELS:
        p = PROB / f'{tag}__{split}.csv'
        if not p.exists(): continue
        d = pd.read_csv(p)
        d['pred'] = (d['p_hate'] >= 0.5).astype(int)
        d['cell'] = [CELLS[(a, q)] for a, q in zip(d['label'], d['pred'])]
        d['lex']  = d['text'].map(lex_hits)
        for cell, g in d.groupby('cell'):
            rows.append({'model': name, 'split': split, 'cell': cell, 'n': len(g),
                         'mean_lex_hits': round(g['lex'].mean(), 2),
                         'pct_zero_lex': round(100 * (g['lex'] == 0).mean(), 1)})
pop = pd.DataFrame(rows)
pop.to_csv(W / 'cell_lexicon_profile.csv', index=False)
display(pop[pop.split == 'within'].pivot(index='model', columns='cell', values='pct_zero_lex'))
print('\npct of posts in each cell containing zero lexicon terms (within-dataset)')

## The spreadsheet to read

One sheet per model so you can work through them one at a time. Within-dataset first,
because that is where he said the behaviour is most puzzling.

Two columns are left blank for you: `your_verdict` and `your_note`. Fill them in as you read.
That turns an impression into evidence, and it is the same method that produced the
category table he liked.

In [ ]:
out = W / 'four_cell_inspection.xlsx'
with pd.ExcelWriter(out, engine='openpyxl') as xl:
    samples.to_excel(xl, sheet_name='all', index=False)
    for tag, name in MODELS:
        sub = samples[(samples.model == name) & (samples.split == 'within')]
        if not sub.empty:
            sub.to_excel(xl, sheet_name=name[:28].replace('/', '-'), index=False)
    cross = samples[samples.split == 'cross']
    if not cross.empty:
        cross.to_excel(xl, sheet_name='cross-dataset', index=False)

samples.to_csv(W / 'four_cell_inspection.csv', index=False)
print('wrote', out)
print('wrote', W / 'four_cell_inspection.csv')
print(f'\n{len(samples)} posts to read. Start with the TF-IDF sheet: 60 posts, '
      '15 from each cell.')

## What to look for while reading

Hunter's question is whether the missed hate posts are genuinely hard, or whether something
systematic is going on. Four things worth deciding as you read:

1. **Would you call it hate speech?** This is the one he cares about most. The earlier
   analysis found 26 of 140 missed posts that you would not. If that holds here, the label
   is wrong, not the model.
2. **Is it political rather than abusive?** 58 of 140 were political criticism. He asked
   directly what the annotators counted as political hate, and whether it had to be unfairly
   abusive.
3. **Does it contain a slur at all?** The lexicon column answers this without guessing.
4. **Does the caught pile look different from the missed pile?** If you cannot tell them
   apart by eye, that is the finding, and it is a defensible one.

Write the answer down as you go. Sixty posts is about forty minutes.